# Function Calling / Tool Use
## 工具调用与Agent系统

<img src="../images/logo.png" width=150>

Function Calling（也称Tool Use）让LLM能够调用外部工具和API，实现从被动生成到主动执行的关键跨越。GPT-4、GPT-4o、Claude 3都原生支持Function Calling。

Function Calling (also called Tool Use) enables LLMs to call external tools and APIs, achieving a key leap from passive generation to active execution. GPT-4, GPT-4o, and Claude 3 all natively support Function Calling.

In [ ]:
import json
from typing import List, Dict, Any, Optional
from dataclasses import dataclass

@dataclass
class Tool:
    """工具定义 / Tool definition"""
    name: str
    description: str
    parameters: Dict[str, Any]  # JSON Schema for parameters
    
    def to_openai_format(self):
        """转换为OpenAI格式 / Convert to OpenAI format"""
        return {
            'type': 'function',
            'function': {
                'name': self.name,
                'description': self.description,
                'parameters': self.parameters
            }
        }

@dataclass
class ToolCall:
    """工具调用请求 / Tool call request"""
    tool_name: str
    arguments: Dict[str, Any]
    call_id: str

# 定义示例工具 / Define example tools
tools = [
    Tool(
        name='get_weather',
        description='Get current weather for a city',
        parameters={
            'type': 'object',
            'properties': {
                'city': {
                    'type': 'string',
                    'description': 'City name'
                },
                'unit': {
                    'type': 'string',
                    'enum': ['celsius', 'fahrenheit'],
                    'default': 'celsius'
                }
            },
            'required': ['city']
        }
    ),
    Tool(
        name='search_database',
        description='Search a database for records',
        parameters={
            'type': 'object',
            'properties': {
                'query': {
                    'type': 'string',
                    'description': 'SQL-like query string'
                },
                'limit': {
                    'type': 'integer',
                    'description': 'Maximum number of results',
                    'default': 10
                }
            },
            'required': ['query']
        }
    ),
    Tool(
        name='send_email',
        description='Send an email to a recipient',
        parameters={
            'type': 'object',
            'properties': {
                'to': {
                    'type': 'string',
                    'description': 'Recipient email address'
                },
                'subject': {
                    'type': 'string',
                    'description': 'Email subject'
                },
                'body': {
                    'type': 'string',
                    'description': 'Email body content'
                }
            },
            'required': ['to', 'subject', 'body']
        }
    )
]

# 打印工具定义 / Print tool definitions
print("Defined Tools:")
for tool in tools:
    print(f"\n  {tool.name}:")
    print(f"    Description: {tool.description}")
    print(f"    Parameters: {json.dumps(tool.parameters, indent=4)}")

# Function Calling解析器
## Function Calling Parser

In [ ]:
import re

class FunctionCallingParser:
    """
    解析LLM输出中的Function Calling请求
    Parse Function Calling requests from LLM output
    """
    def __init__(self, tools: List[Tool]):
        self.tools = {t.name: t for t in tools}
    
    def parse(self, llm_output: str) -> Optional[ToolCall]:
        """
        从LLM输出中解析工具调用
        Parse tool call from LLM output
        
        支持两种格式：
        1. JSON格式: {"name": "func", "arguments": {...}}
        2. OpenAI格式: function_call {name: "func", arguments: {...}}
        """
        # 方法1: 尝试解析JSON / Method 1: Try JSON
        try:
            # 查找JSON对象 / Find JSON object
            json_match = re.search(r'\{[^{}]*"name"[^{}]*"arguments"[^{}]*\}', llm_output, re.DOTALL)
            if json_match:
                data = json.loads(json_match.group())
                if 'name' in data and 'arguments' in data:
                    return ToolCall(
                        tool_name=data['name'],
                        arguments=data['arguments'],
                        call_id=self._generate_id()
                    )
        except json.JSONDecodeError:
            pass
        
        # 方法2: 解析OpenAI格式 / Method 2: Parse OpenAI format
        try:
            func_match = re.search(r'function_call\s*\{([^}]+)\}', llm_output, re.DOTALL)
            if func_match:
                content = func_match.group(1)
                name_match = re.search(r'name:\s*"([^"]+)"', content)
                args_match = re.search(r'arguments:\s*(\{[^}]+\})', content)
                
                if name_match and args_match:
                    return ToolCall(
                        tool_name=name_match.group(1),
                        arguments=json.loads(args_match.group(1)),
                        call_id=self._generate_id()
                    )
        except Exception:
            pass
        
        return None
    
    def _generate_id(self):
        return f"call_{len(self.tools)}_{id(self)}"

# 测试解析器 / Test parser
parser = FunctionCallingParser(tools)

test_outputs = [
    '{"name": "get_weather", "arguments": {"city": "Beijing", "unit": "celsius"}}',
    'I need to check the weather. function_call {name: "get_weather", arguments: {"city": "Shanghai"}}',
    'Let me search for that information. {"name": "search_database", "arguments": {"query": "SELECT * FROM users", "limit": 5}}'
]

for output in test_outputs:
    result = parser.parse(output)
    print(f"Input: {output[:50]}...")
    if result:
        print(f"  -> Tool: {result.tool_name}, Args: {result.arguments}")
    else:
        print(f"  -> No tool call detected")
    print()

# 工具执行器
## Tool Executor

In [ ]:
class ToolExecutor:
    """
    工具执行器 - 实际运行工具
    Tool executor - actually run tools
    """
    def __init__(self):
        self.tools = {}
    
    def register(self, tool_name: str, func):
        """注册工具函数 / Register tool function"""
        self.tools[tool_name] = func
    
    def execute(self, tool_call: ToolCall) -> Any:
        """执行工具调用 / Execute tool call"""
        if tool_call.tool_name not in self.tools:
            return {'error': f"Tool '{tool_call.tool_name}' not found"}
        
        try:
            result = self.tools[tool_call.tool_name](**tool_call.arguments)
            return result
        except Exception as e:
            return {'error': str(e)}

# 定义工具实现 / Define tool implementations
def get_weather_impl(city: str, unit: str = 'celsius') -> dict:
    """模拟天气查询 / Mock weather query"""
    # 实际会调用天气API / In real world, call weather API
    weather_db = {
        'Beijing': {'temp': 22, 'condition': 'Sunny'},
        'Shanghai': {'temp': 25, 'condition': 'Cloudy'},
        'Tokyo': {'temp': 20, 'condition': 'Rainy'},
    }
    
    if city in weather_db:
        data = weather_db[city]
        temp = data['temp'] if unit == 'celsius' else data['temp'] * 9/5 + 32
        return {
            'city': city,
            'temperature': round(temp, 1),
            'unit': unit,
            'condition': data['condition']
        }
    return {'error': f'City {city} not found'}

def search_database_impl(query: str, limit: int = 10) -> dict:
    """模拟数据库查询 / Mock database query"""
    return {
        'query': query,
        'results': [
            {'id': i, 'data': f'record_{i}'} 
            for i in range(min(limit, 5))
        ],
        'count': min(limit, 5)
    }

def send_email_impl(to: str, subject: str, body: str) -> dict:
    """模拟发送邮件 / Mock email sending"""
    return {
        'status': 'sent',
        'to': to,
        'subject': subject,
        'timestamp': '2024-01-15T10:30:00Z'
    }

# 注册工具 / Register tools
executor = ToolExecutor()
executor.register('get_weather', get_weather_impl)
executor.register('search_database', search_database_impl)
executor.register('send_email', send_email_impl)

# 测试执行 / Test execution
print("Tool Execution Tests:")

tool_calls = [
    ToolCall('get_weather', {'city': 'Beijing', 'unit': 'celsius'}, 'test1'),
    ToolCall('search_database', {'query': 'SELECT * FROM users', 'limit': 5}, 'test2'),
    ToolCall('send_email', {'to': 'test@example.com', 'subject': 'Hello', 'body': 'World'}, 'test3')
]

for tc in tool_calls:
    result = executor.execute(tc)
    print(f"\n{tc.tool_name}:")
    print(f"  Arguments: {tc.arguments}")
    print(f"  Result: {result}")

# Agent系统
## Agent System

In [ ]:
class Agent:
    """
    简单的Agent系统：Observation -> Reasoning -> Action
    Simple Agent system: Observation -> Reasoning -> Action
    """
    def __init__(self, tools: List[Tool], executor: ToolExecutor, parser: FunctionCallingParser):
        self.tools = tools
        self.executor = executor
        self.parser = parser
        self.conversation_history = []
    
    def format_tools_for_llm(self) -> str:
        """将工具格式化为LLM可读的描述 / Format tools for LLM"""
        tool_descriptions = []
        tool_descriptions.append("You have access to the following tools:")
        tool_descriptions.append("\n")
        
        for tool in self.tools:
            tool_descriptions.append(f"- {tool.name}: {tool.description}")
            params = tool.parameters.get('properties', {})
            for param_name, param_info in params.items():
                desc = param_info.get('description', '')
                tool_descriptions.append(f"  - {param_name}: {desc}")
            tool_descriptions.append("")
        
        return "\n".join(tool_descriptions)
    
    def process(self, user_input: str, llm_func) -> str:
        """
        处理用户输入
        Process user input
        
        流程：
        1. 构造prompt（包含工具描述）
        2. 调用LLM
        3. 检查是否有工具调用
        4. 执行工具并获取结果
        5. 将结果返回给LLM生成最终回答
        """
        # 保存用户输入 / Save user input
        self.conversation_history.append({'role': 'user', 'content': user_input})
        
        # 构造包含工具的prompt / Construct prompt with tools
        system_prompt = self.format_tools_for_llm()
        
        # 调用LLM（模拟）/ Call LLM (simulated)
        llm_output = llm_func(system_prompt, self.conversation_history)
        
        # 检查工具调用 / Check for tool call
        tool_call = self.parser.parse(llm_output)
        
        if tool_call:
            # 执行工具 / Execute tool
            tool_result = self.executor.execute(tool_call)
            
            # 保存工具调用和结果 / Save tool call and result
            self.conversation_history.append({
                'role': 'assistant',
                'content': llm_output
            })
            self.conversation_history.append({
                'role': 'tool',
                'tool_call_id': tool_call.call_id,
                'tool_name': tool_call.tool_name,
                'content': json.dumps(tool_result)
            })
            
            # 再次调用LLM生成最终回答 / Call LLM again for final answer
            final_response = llm_func(system_prompt, self.conversation_history)
            self.conversation_history.append({
                'role': 'assistant',
                'content': final_response
            })
            
            return final_response
        else:
            # 无工具调用，直接返回 / No tool call, return directly
            self.conversation_history.append({
                'role': 'assistant',
                'content': llm_output
            })
            return llm_output

# 模拟LLM函数 / Mock LLM function
def mock_llm(system_prompt, history):
    """模拟LLM响应 / Mock LLM response"""
    last_user_msg = history[-1]['content'] if history else ''
    
    # 简单规则判断是否需要调用工具 / Simple rule to decide tool usage
    if 'weather' in last_user_msg.lower():
        city = 'Beijing' if 'beijing' in last_user_msg.lower() else 'Shanghai'
        return f'{{"name": "get_weather", "arguments": {{"city": "{city}", "unit": "celsius"}}}}'
    elif 'search' in last_user_msg.lower() or 'database' in last_user_msg.lower():
        return f'{{"name": "search_database", "arguments": {{"query": "SELECT * FROM users", "limit": 5}}}}'
    elif 'email' in last_user_msg.lower():
        return f'{{"name": "send_email", "arguments": {{"to": "user@example.com", "subject": "Hello", "body": "Message"}}}}'
    else:
        return f"Based on the conversation, here is my response to: {last_user_msg}"

# 测试Agent / Test Agent
agent = Agent(tools, executor, parser)

user_inputs = [
    "What's the weather in Beijing?",
    "Search the database for user records",
    "Send an email to test@example.com"
]

print("Agent Conversation Test:")
print("="*50)

for user_input in user_inputs:
    response = agent.process(user_input, mock_llm)
    print(f"User: {user_input}")
    print(f"Agent: {response}")
    print("-"*50)

# ReAct (Reasoning + Acting)
## ReAct Paradigm

In [ ]:
class ReActAgent:
    """
    ReAct Agent: 交替进行推理和行动
    ReAct Agent: Alternates between reasoning and action
    
    循环: Thought -> Action -> Observation -> Thought...
    Loop: Thought -> Action -> Observation -> Thought...
    """
    def __init__(self, tools, executor):
        self.tools = {t.name: t for t in tools}
        self.executor = executor
        self.max_iterations = 5
    
    def run(self, task, llm_func):
        """运行ReAct循环 / Run ReAct loop"""
        observations = []
        thoughts = []
        actions = []
        
        prompt = f"Task: {task}\n\nYou have access to: {list(self.tools.keys())}"
        
        for i in range(self.max_iterations):
            # 1. Thought - 推理下一步
            # 1. Thought - Reason about next step
            context = prompt + "\n\n" + "\n".join([f"Thought: {t}" for t in thoughts])
            context += "\n" + "\n".join([f"Action: {a}" for a in actions])
            context += "\n" + "\n".join([f"Observation: {o}" for o in observations])
            
            thought = llm_func(context, mode='think')
            thoughts.append(thought)
            
            # 检查是否完成 / Check if done
            if 'finish' in thought.lower():
                return {
                    'status': 'completed',
                    'thoughts': thoughts,
                    'actions': actions,
                    'observations': observations
                }
            
            # 2. Action - 选择工具
            # 2. Action - Select tool
            action_str = llm_func(context + f"\nThought: {thought}", mode='action')
            
            try:
                action = json.loads(action_str)
                tool_name = action.get('name')
                tool_args = action.get('arguments', {})
            except:
                observations.append(f"Failed to parse action: {action_str}")
                continue
            
            actions.append(action_str)
            
            # 3. Execute - 执行工具
            # 3. Execute - Execute tool
            if tool_name in self.executor.tools:
                result = self.executor.execute(ToolCall(tool_name, tool_args, f'call_{i}'))
                observations.append(json.dumps(result))
            else:
                observations.append(f"Tool {tool_name} not found")
        
        return {
            'status': 'max_iterations',
            'thoughts': thoughts,
            'actions': actions,
            'observations': observations
        }

# 模拟ReAct LLM / Mock ReAct LLM
def mock_react_llm(context, mode='think'):
    if mode == 'think':
        if 'weather' in context.lower():
            return "I need to check the weather for Beijing. This requires using the get_weather tool."
        else:
            return "I can help with that. Let me finish."
    elif mode == 'action':
        if 'weather' in context.lower():
            return '{"name": "get_weather", "arguments": {"city": "Beijing", "unit": "celsius"}}'
        else:
            return '{"name": "finish", "arguments": {"result": "Task completed"}}'
    return '{}'

# 测试ReAct / Test ReAct
react_agent = ReActAgent(tools, executor)
result = react_agent.run("What's the weather in Beijing?", mock_react_llm)

print("ReAct Agent Result:")
print(f"Status: {result['status']}")
print(f"\nThoughts: {len(result['thoughts'])}")
for i, t in enumerate(result['thoughts']):
    print(f"  {i+1}. {t[:50]}...")
print(f"\nActions: {len(result['actions'])}")
print(f"Observations: {len(result['observations'])}")

# 多工具协作
## Multi-Tool Collaboration

In [ ]:
class ToolChain:
    """
    工具链：顺序执行多个工具
    Tool chain: execute multiple tools in sequence
    """
    def __init__(self):
        self.tools = []
    
    def add(self, tool_name, args_func):
        """
        添加工具到链
        Add tool to chain
        
        args_func: 一个函数，接受前一个工具的输出作为输入
        args_func: A function that takes previous tool's output as input
        """
        self.tools.append((tool_name, args_func))
    
    def execute(self, executor):
        """执行工具链 / Execute tool chain"""
        results = []
        current_input = None
        
        for tool_name, args_func in self.tools:
            # 根据前一个结果构造参数 / Construct args based on previous result
            args = args_func(current_input) if current_input is not None else args_func(None)
            
            # 执行工具 / Execute tool
            tool_call = ToolCall(tool_name, args, f'chain_{len(results)}')
            result = executor.execute(tool_call)
            results.append(result)
            current_input = result
        
        return results

# 示例：多工具协作
# Example: multi-tool collaboration

# 场景：查询天气 -> 根据温度发送邮件
# Scenario: Query weather -> Send email based on temperature

def get_temp_args(previous_result):
    return {'city': 'Beijing', 'unit': 'celsius'}

def send_email_args(previous_result):
    if previous_result and 'temperature' in previous_result:
        temp = previous_result['temperature']
        return {
            'to': 'user@example.com',
            'subject': f'Weather Alert: {temp}°C',
            'body': f'The current temperature is {temp}°C.'
        }
    return {'to': 'user@example.com', 'subject': 'Weather Alert', 'body': 'Temperature info unavailable'}

chain = ToolChain()
chain.add('get_weather', get_temp_args)
chain.add('send_email', send_email_args)

print("Tool Chain Execution:")
results = chain.execute(executor)

print(f"\nStep 1 - get_weather: {results[0]}")
print(f"\nStep 2 - send_email: {results[1]}")

# OpenAI Function Calling 格式
## OpenAI Function Calling Format

In [ ]:
# OpenAI API 格式示例 / OpenAI API format example

def create_openai_function_call_request(user_message, tools):
    """
    创建OpenAI Function Calling请求
    Create OpenAI Function Calling request
    """
    return {
        'model': 'gpt-4-turbo',
        'messages': [
            {'role': 'user', 'content': user_message}
        ],
        'tools': [t.to_openai_format() for t in tools],
        'tool_choice': 'auto'  # or 'none' to disable
    }

def parse_openai_function_call_response(response):
    """
    解析OpenAI Function Calling响应
    Parse OpenAI Function Calling response
    """
    if not response.get('choices'):
        return None
    
    choice = response['choices'][0]
    message = choice['message']
    
    if 'tool_calls' in message:
        tool_calls = []
        for tc in message['tool_calls']:
            func = tc['function']
            tool_calls.append(ToolCall(
                tool_name=func['name'],
                arguments=json.loads(func['arguments']),
                call_id=tc['id']
            ))
        return tool_calls
    
    return None

# 模拟API响应 / Mock API response
mock_response = {
    'choices': [
        {
            'message': {
                'role': 'assistant',
                'tool_calls': [
                    {
                        'id': 'call_abc123',
                        'type': 'function',
                        'function': {
                            'name': 'get_weather',
                            'arguments': '{"city": "Beijing", "unit": "celsius"}'
                        }
                    }
                ]
                'content': None
            }
        }
    ]
}

print("OpenAI Function Calling Example:")
request = create_openai_function_call_request("What's the weather?", tools)
print(f"\nRequest:")
print(json.dumps(request, indent=2))

print(f"\nMock Response:")
tool_calls = parse_openai_function_call_response(mock_response)
if tool_calls:
    for tc in tool_calls:
        print(f"  Tool: {tc.tool_name}")
        print(f"  Arguments: {tc.arguments}")
        print(f"  Call ID: {tc.call_id}")

# Function Calling可视化
## Function Calling Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Function Calling & Agent Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Tool use frequency
ax1 = axes[0]
tools = ['search', 'calculator', 'weather', 'time', 'code_exec']
usage = [450, 320, 280, 150, 120]
colors = plt.cm.Greens(np.linspace(0.4, 0.9, len(tools)))
bars = ax1.barh(tools, usage, color=colors)
ax1.set_xlabel('Call Frequency')
ax1.set_title('Tool使用分布 / Tool Usage Distribution')

# 2. ReAct loop visualization
ax2 = axes[1]
ax2.axis('off')
loop_steps = ['Thought', 'Action', 'Observation', 'Thought', 'Action', 'Final Answer']
y_positions = np.arange(len(loop_steps))
ax2.barh(y_positions, [1]*len(loop_steps), 
         color=['#3498db', '#e74c3c', '#2ecc71', '#3498db', '#e74c3c', '#9b59b6'])
ax2.set_yticks(y_positions)
ax2.set_yticklabels(loop_steps)
ax2.set_title('ReAct循环 / ReAct Loop')

# 3. Agent reasoning steps
ax3 = axes[2]
steps = ['Parse Request', 'Identify Tools', 'Execute Tool 1', 'Execute Tool 2', 'Synthesize Response']
time_ms = [12, 8, 45, 42, 15]
colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(steps)))
bars = ax3.bar(steps, time_ms, color=colors)
ax3.set_ylabel('Time (ms)')
ax3.set_title('Agent处理流程延迟 / Agent Pipeline Latency')
ax3.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../images/function_calling.png', dpi=150, bbox_inches='tight')
plt.show()

print("Function Calling Visualization saved!")

# 总结

| 组件 | 功能 |
|------|------|
| Tool Definition | 定义工具的名称、描述、参数 |
| Function Calling Parser | 解析LLM输出中的工具调用 |
| Tool Executor | 执行实际的工具逻辑 |
| Agent | 协调推理、工具调用、结果整合 |
| ReAct | Thought-Action-Observation循环 |
| Tool Chain | 多工具顺序协作 |

Function Calling是构建复杂Agent系统的基础，使LLM能够执行实际操作而非仅仅生成文本。

In [ ]:
# Function Calling流程详解 / Function Calling Flow Detail
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Tool calling sequence / 工具调用时序
ax1 = axes[0]
ax1.axis('off')

messages = [
    ('User', 'What is the weather in Beijing?'),
    ('Model', 'I will check the weather for you...'),
    ('Tool Call', 'get_weather(city="Beijing")'),
    ('Tool Result', 'Sunny, 25°C'),
    ('Model', 'The weather in Beijing is sunny and 25°C.')
]

colors = ['#f39c12', '#3498db', '#e74c3c', '#2ecc71', '#3498db']
y_positions = np.arange(len(messages))[::-1]

for i, (role, msg) in enumerate(messages):
    ax1.text(0.05, y_positions[i], role + ':', fontsize=9, fontweight='bold', va='center')
    ax1.text(0.15, y_positions[i], msg[:40], fontsize=8, va='center')

ax1.set_xlim(0, 1)
ax1.set_ylim(-0.5, len(messages)-0.5)
ax1.set_title('Function Calling Sequence
(User -> Model -> Tool -> Model -> User)')

# 2. JSON Schema structure / JSON Schema结构
ax2 = axes[1]
ax2.axis('off')

schema = {
    'name': 'get_weather',
    'description': 'Get current weather for a city',
    'parameters': {
        'type': 'object',
        'properties': {
            'city': {'type': 'string', 'description': 'City name'},
            'unit': {'type': 'string', 'enum': ['celsius', 'fahrenheit']}
        },
        'required': ['city']
    }
}

lines = [
    '{',
    '  "name": "get_weather",',
    '  "description": "Get weather...",',
    '  "parameters": {',
    '    "type": "object",',
    '    "properties": {',
    '      "city": {"type": "string"},',
    '      ...',
    '    }',
    '  }',
    '}'
]

for i, line in enumerate(lines):
    ax2.text(0.1, len(lines)-i, line, fontsize=8, fontfamily='monospace')

ax2.set_xlim(0, 1)
ax2.set_ylim(0, len(lines))
ax2.set_title('Tool Definition JSON Schema')

# 3. ReAct loop visualization
ax3 = axes[2]
ax3.axis('off')

loop_steps = [
    ('Thought', 'Think about what tool to use', '#3498db'),
    ('Action', 'Call the tool', '#e74c3c'),
    ('Observation', 'Get tool result', '#2ecc71'),
    ('Thought', 'Analyze result', '#3498db'),
    ('Action', 'Generate final answer', '#e74c3c'),
    ('Response', 'Return to user', '#9b59b6')
]

for i, (step, desc, color) in enumerate(loop_steps):
    y = len(loop_steps) - i - 1
    circle = plt.Circle((0.3, y), 0.35, facecolor=color, edgecolor='black', linewidth=1.5)
    ax3.add_patch(circle)
    ax3.text(0.3, y, step, ha='center', va='center', fontsize=8, fontweight='bold', color='white')
    ax3.text(0.8, y, desc, ha='left', va='center', fontsize=8)

ax3.set_xlim(-0.2, 1.5)
ax3.set_ylim(-0.5, len(loop_steps)-0.5)
ax3.set_title('ReAct Loop
(Reasoning + Acting)')

plt.tight_layout()
plt.savefig('../images/function_calling_flow.png', dpi=150, bbox_inches='tight')
plt.show()

print("Function calling flow visualization saved!")

# 已实现 / Implemented

本notebook已完整实现以下内容：

1. **工具定义** - OpenAI格式的工具schema
2. **FunctionCallingParser** - 解析和调用函数
3. **ToolExecutor** - 工具执行环境
4. **Agent系统** - 完整的Agent循环
5. **ReAct范式** - Thought/Action/Observation循环

## 扩展阅读 / Further Reading

| 主题 | 说明 | 推荐资源 |
|------|------|----------|
| **OpenAI Function Calling** | 官方API | [OpenAI Docs](https://platform.openai.com/docs/guides/function-calling) |
| **Streamlit UI** | Web演示界面 | [Streamlit](https://streamlit.io/) |
| **多Agent系统** | Agent协作 | [AutoGen](https://microsoft.github.io/autogen/) |
| **Toolformer** | Tool Learning基础 | [Toolformer](https://arxiv.org/abs/2302.04761) |
| **ReAct原论文** | Reason + Act范式 | [ReAct Paper](https://arxiv.org/abs/2210.03629) |
